# Prosody CKA pipeline — verification notebook

This notebook runs the colm pipeline step-by-step so you can verify:
1. **Expresso** data loading and parallel same-text groups
2. **ESD** data loading and parallel emotion groups
3. **CKA** (Centered Kernel Alignment) on dummy features

Run from the **parent of colm** (e.g. `bkoduru/`) or ensure `colm` is on `PYTHONPATH`.

## 1. Setup: path and imports

In [1]:
import os
import sys

# Add colm project folder so "import colm" works (if running from parent of colm)
_here = os.path.abspath(os.getcwd())
if _here.endswith('colm'):
    _colm_root = _here
else:
    _colm_root = os.path.join(_here, 'colm')
if os.path.isdir(_colm_root) and _colm_root not in sys.path:
    sys.path.insert(0, _colm_root)
print('Colm root:', _colm_root)

import numpy as np
from colm.config import EXPRESSO_ROOT, ESD_ROOT, EXPRESSO_READ_STYLES, ESD_EMOTIONS
from colm.data.expresso import (
    load_expresso_transcriptions,
    get_expresso_audio_path,
    build_expresso_parallel_groups,
)
from colm.data.esd import build_esd_parallel_groups, get_esd_audio_path
from colm.data.pair_manifests import load_expresso_pairs_manifest, load_esd_pairs_manifest, iter_pairs_for_cka
from colm.cka import linear_cka, kernel_cka

print('Expresso root:', EXPRESSO_ROOT)
print('ESD root:', ESD_ROOT)
print('Expresso exists:', os.path.isdir(EXPRESSO_ROOT))
print('ESD exists:', os.path.isdir(ESD_ROOT))

Colm root: /ocean/projects/cis220031p/bkoduru/colm
Expresso root: /ocean/projects/cis220031p/bkoduru/expresso
ESD root: /ocean/projects/cis220031p/bkoduru/Emotion Speech Dataset
Expresso exists: True
ESD exists: True


## 2. Expresso: load transcriptions and build parallel groups

In [2]:
# Optional: override roots if your data is elsewhere
EXPRESSO_ROOT_USE = os.environ.get('EXPRESSO_ROOT', EXPRESSO_ROOT)
ESD_ROOT_USE = os.environ.get('ESD_ROOT', ESD_ROOT)

transcriptions = load_expresso_transcriptions(EXPRESSO_ROOT_USE)
print(f'Loaded {len(transcriptions)} Expresso read transcriptions')
print('Sample (first 3):')
for i, (uid, text) in enumerate(transcriptions.items()):
    if i >= 3:
        break
    print(f'  {uid} -> {repr(text[:60])}...')

Loaded 11615 Expresso read transcriptions
Sample (first 3):
  ex01_confused_00001 -> 'Why are you beating up my jukebox?'...
  ex01_confused_00002 -> 'I have to stop you.'...
  ex01_confused_00003 -> "Monday, there's gonna be haze, but Tuesday, look for thunder"...


In [3]:
parallel_groups, emphasis_groups = build_expresso_parallel_groups(
    root=EXPRESSO_ROOT_USE,
    min_styles_per_group=2,
)
print(f'Parallel groups (same sentence, multiple styles): {len(parallel_groups)}')
print(f'Emphasis subset groups: {len(emphasis_groups)}')

Parallel groups (same sentence, multiple styles): 1520
Emphasis subset groups: 396


In [4]:
# Show first 2 groups and verify files exist
for gidx, group in enumerate(parallel_groups[:2]):
    print(f'--- Group {gidx + 1} ---')
    print(f'  text: {group["text_raw"][:70]}...')
    for u in group['utterances']:
        exists = os.path.isfile(u['path'])
        print(f'    {u["style"]}: {"OK" if exists else "MISSING"} {u["path"]}')
    print()

--- Group 1 ---
  text: Why are you beating up my jukebox?...
    confused: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/confused/base/ex01_confused_00001.wav
    default: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/default/base/ex01_default_00001.wav
    enunciated: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/enunciated/base/ex01_enunciated_00001.wav
    happy: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/happy/base/ex01_happy_00001.wav
    laughing: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/laughing/base/ex01_laughing_00001.wav
    sad: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/sad/base/ex01_sad_00001.wav
    whisper: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_48khz/read/ex01/whisper/base/ex01_whisper_00001.wav

--- Group 2 ---
  text: I have to stop you....
    confused: OK /ocean/projects/cis220031p/bkoduru/expresso/audio_

## 3. ESD: build parallel emotion groups

In [5]:
esd_groups = build_esd_parallel_groups(
    root=ESD_ROOT_USE,
    min_emotions_per_group=2,
)
print(f'ESD parallel groups (same sentence, multiple emotions): {len(esd_groups)}')
if esd_groups:
    g = esd_groups[0]
    print(f'Sample group: speaker={g["speaker"]}, sentence_index={g["sentence_index"]}')
    print(f'  text: {g["text"][:50]}...')
    for u in g['utterances']:
        exists = os.path.isfile(u['path'])
        print(f'    {u["emotion"]}: {"OK" if exists else "MISSING"}')

ESD parallel groups (same sentence, multiple emotions): 7000
Sample group: speaker=0001, sentence_index=0
  text: 打远一看，它们的确很是美丽，...
    Neutral: OK
    Angry: OK


## 4. CKA: sanity check on random matrices

In [6]:
np.random.seed(42)
n, d1, d2 = 100, 64, 128
X = np.random.randn(n, d1).astype(np.float64)
Y = np.random.randn(n, d2).astype(np.float64)

c_linear = linear_cka(X, Y)
c_kernel = kernel_cka(X, Y)
print(f'Linear CKA (random X,Y): {c_linear:.4f}')
print(f'Kernel CKA (random X,Y): {c_kernel:.4f}')

# Same matrix should give 1.0
c_same = linear_cka(X, X)
print(f'Linear CKA (X,X): {c_same:.4f} (expect 1.0)')

Linear CKA (random X,Y): 0.4728
Kernel CKA (random X,Y): 0.6490
Linear CKA (X,X): 1.0000 (expect 1.0)


## 5. Run CKA on Expresso groups (dummy features)

In [7]:
def dummy_extract(audio_paths, feat_dim=64):
    return [np.random.randn(50, feat_dim).astype(np.float32) for _ in audio_paths]

max_groups = 10
feat_dim = 64
groups = parallel_groups[:max_groups]
ckas = []
for g in groups:
    paths = [u['path'] for u in g['utterances']]
    reprs = dummy_extract(paths, feat_dim)
    n = len(reprs)
    for i in range(n):
        for j in range(i + 1, n):
            c = linear_cka(reprs[i], reprs[j])
            ckas.append(c)

ckas = np.array(ckas)
print(f'Groups: {len(groups)}, Pairs: {len(ckas)}')
print(f'CKA mean: {np.mean(ckas):.4f}, std: {np.std(ckas):.4f}')
print('(With random features, CKA is ~0.5–0.6; with real encoder/projector features you will see different values.)')

Groups: 10, Pairs: 210
CKA mean: 0.5610, std: 0.0150
(With random features, CKA is ~0.5–0.6; with real encoder/projector features you will see different values.)


## 6. Optional: run same on ESD and save results

In [8]:
ckas_esd = np.array([])
if esd_groups:
    esd_subset = esd_groups[:max_groups]
    ckas_esd_list = []
    for g in esd_subset:
        paths = [u['path'] for u in g['utterances']]
        reprs = dummy_extract(paths, feat_dim)
        n = len(reprs)
        for i in range(n):
            for j in range(i + 1, n):
                ckas_esd_list.append(linear_cka(reprs[i], reprs[j]))
    ckas_esd = np.array(ckas_esd_list)
    print(f'ESD: Groups {len(esd_subset)}, Pairs {len(ckas_esd)}, CKA mean {np.mean(ckas_esd):.4f}')
else:
    print('No ESD groups (path missing or empty).')

ESD: Groups 10, Pairs 10, CKA mean 0.5627


In [9]:
# Save to .npz for later use
out = {
    'expresso_cka_mean': float(np.mean(ckas)) if len(ckas) else np.nan,
    'expresso_cka_std': float(np.std(ckas)) if len(ckas) else np.nan,
    'expresso_n_groups': len(groups),
    'expresso_n_pairs': len(ckas),
}
if esd_groups and len(ckas_esd):
    out['esd_cka_mean'] = float(np.mean(ckas_esd))
    out['esd_n_pairs'] = len(ckas_esd)
np.savez('verify_cka_results.npz', **out)
print('Saved verify_cka_results.npz')

Saved verify_cka_results.npz


## 7. Run CKA from manifest JSONs (optional)

Use your pre-built pair lists (e.g. `interspeech/expresso_local_2.json`, `esp/esp_local.json`) so you don't build groups from scratch. Set paths and `max_pairs` as needed.

In [ ]:
# Paths to your manifest JSONs (adjust if needed)
_project = os.path.dirname(_colm_root)  # parent of colm
EXPRESSO_MANIFEST = os.path.join(_project, "interspeech", "expresso_local_2.json")
ESD_MANIFEST = os.path.join(_project, "interspeech", "esp", "esp_local.json")

max_pairs_manifest = 200  # cap for a quick run; set to None to use all pairs

# Expresso manifest
if os.path.isfile(EXPRESSO_MANIFEST):
    expresso_pairs = load_expresso_pairs_manifest(EXPRESSO_MANIFEST)
    subset = expresso_pairs[:max_pairs_manifest] if max_pairs_manifest else expresso_pairs
    ckas_ex = []
    for p in subset:
        reprs = dummy_extract([p["audio1_path"], p["audio2_path"]], feat_dim)
        ckas_ex.append(linear_cka(reprs[0], reprs[1]))
    ckas_ex = np.array(ckas_ex)
    print(f"Expresso manifest: {len(expresso_pairs)} pairs total, ran {len(ckas_ex)}, CKA mean = {np.mean(ckas_ex):.4f}")
else:
    print(f"Expresso manifest not found: {EXPRESSO_MANIFEST}")

# ESD manifest
if os.path.isfile(ESD_MANIFEST):
    esd_pairs = load_esd_pairs_manifest(ESD_MANIFEST)
    subset_esd = esd_pairs[:max_pairs_manifest] if max_pairs_manifest else esd_pairs
    ckas_esd_manifest = []
    for p in subset_esd:
        reprs = dummy_extract([p["audio1_path"], p["audio2_path"]], feat_dim)
        ckas_esd_manifest.append(linear_cka(reprs[0], reprs[1]))
    ckas_esd_manifest = np.array(ckas_esd_manifest)
    print(f"ESD manifest: {len(esd_pairs)} pairs total, ran {len(ckas_esd_manifest)}, CKA mean = {np.mean(ckas_esd_manifest):.4f}")
else:
    print(f"ESD manifest not found: {ESD_MANIFEST}")